In [ ]:
#Import necessary packages
import pandas as pd
import seaborn as sns
sns.set_style("whitegrid")
import pandas as pd
import numpy as np 
import scipy.stats as stats
from collections import Counter
import matplotlib.pyplot as plt
import umap
import matplotlib
import mygene
%matplotlib inline
import pickle
import scipy.sparse as sp
import sklearn
import random
import scanpy as sc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.cluster import MiniBatchKMeans
from xgboost import XGBClassifier
# import sentence_transformers
plt.style.use('ggplot')
#plt.style.use('seaborn-v0_8-dark-palette')
plt.rcParams['axes.facecolor'] = 'white'
# plt.rcParams.update({
#     "text.usetex": False,
#     "font.family": "Helvetica"
# })
import matplotlib_inline
import scib_metrics
matplotlib_inline.backend_inline.set_matplotlib_formats('retina')
import openai
# use hnswlib for NN classification
try:
    import hnswlib
    hnswlib_imported = True
except ImportError:
    hnswlib_imported = False
    print("hnswlib not installed! We highly recommend installing it for fast similarity search.")
    print("To install it, run: pip install hnswlib")
from scipy.stats import mode

In [ ]:

import os
import torch
import random

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

In [ ]:
# Here we consider steps to read different datasets.

adata = sc.read("../Clustring_data/ATAA/gse155468.h5ad")
adata.obs_names_make_unique()

In [ ]:
import numpy as np
import scipy.sparse as sp
import anndata as ad
import pickle

def compute_embeddings2(adata, path, mode):
    """
    Compute cell embeddings by combining expression with per-gene embeddings.

    Parameters
    ----------
    adata : AnnData
        adata.X: (cells x genes), dense or sparse.
        adata.var must contain gene names (tries 'gene_symbol', 'gene_name', then index).
    path : str
        Pickle path storing a dict {gene_name: embedding_vector}.
    mode : {'wa','aa'}
        'wa' = row-normalised weighted average of gene embeddings per cell.
        'aa' = simple average of gene embeddings present (X @ G) / n_genes.
    casefold : bool
        If True, case-normalise gene names in both adata.var and the embedding dict.

    Returns
    -------
    np.ndarray, shape (n_cells, d)
    """

    # ---- 1) Load gene embeddings ----
    with open(path, "rb") as fp:
        gene_emb = pickle.load(fp)    # e.g., {'TP53': np.array([...]), ...}


    # ---- 2) Pick gene name field ----
    if "gene_symbol" in adata.var.columns:
        genes = adata.var["gene_symbol"].astype(str).tolist()
    elif "gene_name" in adata.var.columns:
        genes = adata.var["gene_name"].astype(str).tolist()
    else:
        genes = adata.var_names.astype(str).tolist()


    # ---- 3) Build lookup matrix G (genes x d) aligned to adata.var order ----
    # infer embedding dim from the first vector
    example_vec = next(iter(gene_emb.values()))
    d = int(np.asarray(example_vec).shape[-1])

    G = np.zeros((len(genes), d), dtype=np.float32)
    missing = 0
    for i, g in enumerate(genes):
        vec = gene_emb.get(g)
        if vec is not None:
            G[i] = np.asarray(vec, dtype=np.float32)
        else:
            missing += 1
    print(f"Unable to match {missing} of {len(genes)} genes.")

    # ---- 4) Compute embeddings ----
    X = adata.X

    if mode == "wa":
        # Row-normalise X (per cell) then multiply by G
        if sp.issparse(X):
            X = X.tocsr(copy=True)
            row_sums = np.asarray(X.sum(axis=1)).ravel()
            # avoid divide-by-zero
            inv = np.divide(1.0, row_sums, out=np.zeros_like(row_sums, dtype=np.float32), where=row_sums != 0)
            # scale rows
            X_norm = sp.diags(inv, format="csr") @ X  # (cells x genes)
            out = X_norm @ G                          # (cells x d)
            out = np.asarray(out, dtype=np.float32)
        else:
            X = np.asarray(X, dtype=np.float32)
            row_sums = X.sum(axis=1)
            row_sums[row_sums == 0] = 1e-10
            X_norm = X * (1.0 / row_sums)[:, None]
            out = X_norm @ G

    elif mode == "aa":
        # Simple average over genes (does NOT normalise per cell)
        # (cells x genes) @ (genes x d) / n_genes
        if sp.issparse(X):
            out = (X @ G) / float(len(genes))
            out = np.asarray(out, dtype=np.float32)
        else:
            out = (np.asarray(X, dtype=np.float32) @ G) / float(len(genes))
    else:
        raise ValueError("mode must be 'wa' or 'aa'")

    return out  # shape: (n_cells, d)
scELMo_emebed_test = compute_embeddings2(adata, 
                                          "../ensem_emb_gpt3.5all_new.pickle",
                                          "wa")

## Save embeddings

In [ ]:
output_path = '../embeddings/ATAA_with_zeroshot_embeddings.h5ad'

adata.write_h5ad(output_path)

print(f"Embeddings saved to {output_path} in obsm['emb']")

## UMAP Visualisation of Embeddings

In [ ]:
adata.obsm['X_gpt3.5'] = scELMo_emebed_test

sc.pp.neighbors(adata, use_rep='X_gpt3.5', random_state=seed)
sc.tl.umap(adata, random_state=seed)
plt.rcParams['figure.figsize'] = (6, 6)
sc.pl.umap(adata, color='celltype', palette='Paired', title='scELMo', show=False)
ax = plt.gca()
handles, labels = ax.get_legend_handles_labels()
plt.legend(
    handles, labels,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    ncol=1,  # <-- this forces single-column legend
    fontsize='small',
    frameon=False
)
file_path = '../figures/umap_zero_shot_clustering_ATAA.svg'
plt.savefig(file_path, dpi=500, bbox_inches='tight')
plt.show()

## Clustering

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.decomposition import PCA


def clustering(adata, n_clusters=7, key='X_gpt3.5', method='leiden', start=0.1, end=3.0, increment=0.01):
    """\
    Spatial clustering based the learned representation.

    Returns
    -------
    None.

    """
    
    pca = PCA(n_components=20, random_state=seed) 
    embedding = pca.fit_transform(adata.obsm[key].copy())
    adata.obsm['emb_pca'] = embedding
    
    if method == 'leiden':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['leiden']
    elif method == 'louvain':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['louvain'] 
       
    

def extract_top_value(map_matrix, retain_percent = 0.1): 
    '''\
    Filter out cells with low mapping probability

    Parameters
    ----------
    map_matrix : array
        Mapped matrix with m spots and n cells.
    retain_percent : float, optional
        The percentage of cells to retain. The default is 0.1.

    Returns
    -------
    output : array
        Filtered mapped matrix.

    '''

    #retain top 1% values for each spot
    top_k  = retain_percent * map_matrix.shape[1]
    output = map_matrix * (np.argsort(np.argsort(map_matrix)) >= map_matrix.shape[1] - top_k)
    
    return output 
    
def search_res(adata, n_clusters, method='leiden', use_rep='emb', start=0.1, end=3.0, increment=0.01):
    '''\
    Searching corresponding resolution according to given cluster number
    
    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Targetting number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.    
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float 
        The end value for searching.
    increment : float
        The step size to increase.
        
    Returns
    -------
    res : float
        Resolution.
        
    '''
    print('Searching resolution...')
    label = 0
    sc.pp.neighbors(adata, n_neighbors=50, use_rep=use_rep, random_state=seed)
    for res in sorted(list(np.arange(start, end, increment)), reverse=True):
        if method == 'leiden':
           sc.tl.leiden(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['leiden']).leiden.unique())
           print('resolution={}, cluster number={}'.format(res, count_unique))
        elif method == 'louvain':
           sc.tl.louvain(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['louvain']).louvain.unique()) 
           print('resolution={}, cluster number={}'.format(res, count_unique))
        if count_unique == n_clusters:
            label = 1
            break

    assert label==1, "Resolution is not found. Please try bigger range or smaller step!." 
       
    return res    

In [ ]:

n_clusters = 11
tool='leiden'

clustering(adata, n_clusters, key='X_gpt3.5', method=tool, start=0.1, end=0.45, increment=0.01)
labels = adata.obs['leiden'].astype(int)

## ARI, NMI & Silhouette Scores

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

ari_score = adjusted_rand_score(adata.obs['leiden'].to_numpy(), adata.obs['celltype'].to_numpy())
nmi_score = normalized_mutual_info_score(adata.obs['leiden'].to_numpy(), adata.obs['celltype'].to_numpy())
sil_score = silhouette_score(adata.obsm['X_gpt3.5'], adata.obs['leiden'].astype(int))


print(f"'ari': {ari_score}, 'nmi': {nmi_score}, 'sil': {sil_score}")


## Save results to npz file

In [ ]:
Model_name='scELMo'
step='zero_shot'
dataset='BMMC'

In [ ]:

import numpy as np

ARI, NMI, SIL = float(ari_score), float(nmi_score), float(sil_score)

# --- SAVE (one compact file per model) ---
np.savez_compressed(
    f"./benchmarking_results/{Model_name}_{step}_clusters_{dataset}.npz",
    labels=data.obs['celltype'],      
    embeddings=data.obsm['emb'],
    ARI=ARI, NMI=NMI, SIL=SIL,
)